# Coherent contaminants and the EACF veto

This tutorial compares a stochastic p-mode comb with a coherent periodic contaminant under the same observing window. The key idea is that a coherent signal occupies fewer Fourier bins, even when the window function creates aliases.

In [ ]:
import numpy as np

from urdr import (
    CoherentSignalConfig,
    SimulationConfig,
    TimeSeries,
    add_coherent_signal,
    benchmark_coherent_veto,
    coherence_diagnostics,
    simulate_time_series,
)

## Build an exact observing window

Missing cadences stay on the uniform grid. This lets the same window function act on the oscillation, contaminant, and null simulations.

In [ ]:
size = 4096
time_days = np.arange(size) * (120.0 / 86400.0)
observed = np.ones(size, dtype=bool)
observed[900:1020] = False
template = TimeSeries(time_days, np.zeros(size), observed)

simulation = SimulationConfig(
    white_noise_sigma=0.15,
    granulation_amplitude=0.25,
    granulation_timescale_days=0.2,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.0,
)
template.window.diagnostics(simulation.delta_nu_uhz)

## Compare stochastic and coherent signals

The contaminant is added to the same noise-plus-granulation realisation used by the null. Independent harmonic phases make the simulation reproducible without forcing an artificial phase alignment.

In [ ]:
rng = np.random.default_rng(12)
oscillator = simulate_time_series(template.window, simulation, rng)
noise = simulate_time_series(
    template.window,
    simulation,
    np.random.default_rng(12),
    include_oscillations=False,
)
coherent = add_coherent_signal(
    noise,
    CoherentSignalConfig(1000.0, 1.0, harmonics=1),
    np.random.default_rng(13),
)

diagnostics = {
    "oscillator": coherence_diagnostics(oscillator, 1000.0, 500.0),
    "coherent": coherence_diagnostics(coherent, 1000.0, 500.0),
}
diagnostics

A coherent source should usually have a larger `maximum_bin_fraction`, fewer `effective_bins`, and lower `spectral_entropy`. These are diagnostics rather than universal cuts: the observing window and target noise change their distributions.

## Calibrate the veto for this target

The EACF threshold comes from clean null simulations. A separate concentration threshold retains a requested fraction of detected oscillation injections. The small realisation count below keeps the tutorial quick; production calibration should normally use at least 128.

In [ ]:
metrics = benchmark_coherent_veto(
    window=template.window,
    simulation=simulation,
    contaminants={
        "single_line": CoherentSignalConfig(1000.0, 1.0),
        "three_harmonics": CoherentSignalConfig(
            333.3, 1.0, harmonics=3
        ),
    },
    centre_frequencies_uhz=np.array([900.0, 1000.0, 1100.0]),
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=np.linspace(90.0, 110.0, 9),
    realizations=8,
    target_false_positive_rate=0.25,
    target_signal_retention=0.9,
    max_lag_seconds=15_000.0,
    seed=42,
)
metrics